In [4]:
# Implementation - PCS
# Heart Disease Dataset (heart.csv)

import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Models
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score

# PCA
from sklearn.decomposition import PCA

# ---------------------------------------------------
# Step 1: Load Dataset
# ---------------------------------------------------
data = pd.read_csv("/content/heart (1).csv")

print(data.head())

# ---------------------------------------------------
# Step 2: Separate Features and Target
# ---------------------------------------------------
X = data.drop("HeartDisease", axis=1)   # target column assumed
y = data["HeartDisease"]

# ---------------------------------------------------
# Step 3: Find Categorical Columns
# ---------------------------------------------------
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

print("Categorical Columns:", cat_cols)
print("Numerical Columns:", num_cols)

# ---------------------------------------------------
# Step 4: Encoding
# Label Encoding + One Hot Encoding
# ---------------------------------------------------
for col in cat_cols:
    if X[col].nunique() == 2:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

cat_cols = X.select_dtypes(include=['object']).columns

ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

X = ct.fit_transform(X)

# ---------------------------------------------------
# Step 5: Train Test Split
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------
# Step 6: Feature Scaling
# ---------------------------------------------------
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# ---------------------------------------------------
# Step 7: Train Different Models
# ---------------------------------------------------
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier()
}

print("Accuracy Before PCA")
best_model = None
best_acc = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(name, ":", round(acc*100,2), "%")

    if acc > best_acc:
        best_acc = acc
        best_model = name

print("\nBest Model =", best_model)

# ---------------------------------------------------
# Step 8: Apply PCA
# ---------------------------------------------------
pca = PCA(n_components=0.95)   # Keep 95% variance

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Original Features :", X_train.shape[1])
print("Reduced Features :", X_train_pca.shape[1])

# ---------------------------------------------------
# Step 9: Retrain Models After PCA
# ---------------------------------------------------
print("\nAccuracy After PCA")

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, pred)
    print(name, ":", round(acc*100,2), "%")

# ---------------------------------------------------
# Conclusion
# ---------------------------------------------------
print("\nPCA reduces dimensions and training time.")
print("Sometimes accuracy decreases slightly,")
print("but computation becomes faster.")

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  
Categorical Columns: Index(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], dtype='object')
Numerical Columns: Index(['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak'],